# MODEL

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor


df = pd.read_csv("data/TABLEAU/file_TABLEAU_V6(quartieri_standard).csv", sep=';', decimal=',')
df = df[df["price"] < 600000]
df = df[df["size_m2"] < 200]
df = df[df["distanza_universita_m"] < 6700]
df = df.drop(columns=["distanza_ospedale_m", "distanza_stazione_m", "distanza_universita_m", "dist_metro_m", "distanza_asilo_m", "distanza_scuola_superiore_m", "dist_bus_m", "dist_tram_m", "distanza_scuola_media_m", "distanza_scuola_elementare_m", "Bathrooms", "via", "civico", "citta", "lat", "lon", "minuti_metro", "minuti_tram", "minuti_bus", "ospedale_piu_vicino", "tempo_piedi_ospedale_min"
                     , "universita_piu_vicino", "tempo_piedi_universita_min", "stazione_piu_vicino", "tempo_piedi_stazione_min", "asilo_piu_vicino", "tempo_piedi_asilo_min", "scuola_elementare_piu_vicino", "tempo_piedi_scuola_elementare_min"
                     , "scuola_media_piu_vicino", "tempo_piedi_scuola_media_min", "scuola_superiore_piu_vicino", 
                     "tempo_piedi_scuola_superiore_min", "data_aggiornamento", "coefficient", "esposizione", "infissi_vetro", "infissi_materiale"])
# -------------------------
# 1. COPY DATASET
# -------------------------
df_model = df.copy()

# -------------------------
# 2. ENCODING CATEGORICAL VARIABLES
# -------------------------
label_encoders = {}

for col in df_model.select_dtypes(include="object").columns:
    if col == "quartiere":
        continue

    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le


df_model = df_model.drop(columns=["quartiere"])

# -------------------------
# 3. SPLIT X / y (LOG TARGET)
# -------------------------
X = df_model.drop(columns=["price"])
y = np.log1p(df_model["price"])

# -------------------------
# 4. TRAIN / TEST SPLIT
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# -------------------------
# 5. TRAIN XGBOOST MODEL
# -------------------------
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------
# 6. PREDICTION + EVALUATION
# -------------------------
y_pred = np.expm1(model.predict(X_test))
y_test = np.expm1(y_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")


# -------------------------
# ADDITIONAL METRICS
# -------------------------
from sklearn.metrics import mean_squared_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# MAPE (errore percentuale)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

# Median Absolute Error (robusto agli outlier)
median_error = np.median(np.abs(y_test - y_pred))

print("\nAdditional Metrics:\n")
print(f"RMSE  : {rmse:.2f}")
print(f"R²    : {r2:.4f}")
print(f"MAPE  : {mape:.2f}%")
print(f"Median Absolute Error: {median_error:.2f}")


# -------------------------
# PLOT REAL vs PREDICTED
# -------------------------
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.xlabel("Real Price")
plt.ylabel("Predicted Price")
plt.title("Real vs Predicted Prices")

# linea perfetta
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--")

plt.show()

# -------------------------
# 7. FEATURE IMPORTANCE
# -------------------------
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nFeature Importance:\n")
print(importance_df)

# -------------------------
# 8. PLOT FEATURE IMPORTANCE
# -------------------------
plt.figure(figsize=(10,6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.show()



# -------------------------
# 9. SAVE MODEL + ARTIFACTS
# -------------------------
os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/xgb_model.pkl")
joblib.dump(list(X.columns), "model/features.pkl")
joblib.dump(label_encoders, "model/label_encoders.pkl")

print("\n✅ MODEL + ARTIFACTS SAVED SUCCESSFULLY")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe0 in position 1004: invalid continuation byte

# PREDICTIONS

In [4]:
import pandas as pd
import numpy as np
import joblib


# -------------------------
# 1. LOAD MODEL + ARTIFACTS
# -------------------------
model = joblib.load("model/xgb_model.pkl")
features = joblib.load("model/features.pkl")
label_encoders = joblib.load("model/label_encoders.pkl")


# -------------------------
# 2. PREDICTION FUNCTION
# -------------------------
def predict_price(input_dict):
    """
    input_dict = dizionario con le feature del modello
    """

    df = pd.DataFrame([input_dict])

    # -------------------------
    # RIMUOVI QUARTIERE (non usato nel modello)
    # -------------------------
    if "quartiere" in df.columns:
        df = df.drop(columns=["quartiere"])


    # -------------------------
    # LABEL ENCODING VARIABILI CATEGORICHE
    # -------------------------
    for col, le in label_encoders.items():
        if col in df.columns:
            df[col] = df[col].astype(str)

            # gestione valori mai visti
            df[col] = df[col].apply(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )

            df[col] = le.transform(df[col])


    # -------------------------
    # ALLINEAMENTO FEATURE
    # -------------------------
    df = df.reindex(columns=features)

    # eventuali colonne mancanti → 0
    df = df.fillna(0)


    # -------------------------
    # PREDICTION
    # -------------------------
    pred_log = model.predict(df)
    pred = np.expm1(pred_log)

    return float(pred[0])


# -------------------------
# 3. TEST EXAMPLE
# -------------------------
if __name__ == "__main__":

    new_house = {
        "size_m2": 35,
        "numero_locali": 1,
        "quartiere_pulito": "San Salvario",  
        "elevator_final": 1,
        "Balcony": 0,
        "tipo_immobile": "Monolocale",
        "energy_class": "B",
        "floor": 0,
        "luxury_score": 4
    }

    price = predict_price(new_house)

    print("\n🏠 Prezzo stimato:", round(price, 2), "€")


🏠 Prezzo stimato: 96610.83 €
